# 04 - Deploy Model to RHOAI Serving

**Elyra Pipeline Node 4** — Deploy the trained model from S3 to RHOAI model serving (KServe) using the Kubernetes Python client.

**Required (set via Elyra node properties or env)**:
- `S3_BUCKET`, `S3_PREFIX` or `S3_MODEL_URI` — where model was copied by node 03
- `INFERENCE_SERVICE_NAMESPACE` — OpenShift project for serving
- `INFERENCE_SERVICE_NAME` — optional, default `isolation-forest`

In [ ]:
import os

# From Elyra node properties: set these in the pipeline editor for the Deploy node
S3_MODEL_URI = os.getenv("S3_MODEL_URI", "")
S3_BUCKET = os.getenv("S3_BUCKET", "")
S3_PREFIX = os.getenv("S3_PREFIX", "models/isolation-forest/")
INFERENCE_SERVICE_NAME = os.getenv("INFERENCE_SERVICE_NAME", "isolation-forest")
INFERENCE_SERVICE_NAMESPACE = os.getenv("INFERENCE_SERVICE_NAMESPACE", "")

if not S3_MODEL_URI and S3_BUCKET:
    S3_MODEL_URI = f"s3://{S3_BUCKET}/{S3_PREFIX.rstrip('/')}/"

if not S3_MODEL_URI:
    raise ValueError("S3_MODEL_URI or S3_BUCKET must be set (model in S3 from node 03)")
if not INFERENCE_SERVICE_NAMESPACE:
    raise ValueError("INFERENCE_SERVICE_NAMESPACE must be set (OpenShift project)")

print(f"Model URI: {S3_MODEL_URI}")
print(f"InferenceService: {INFERENCE_SERVICE_NAME} in {INFERENCE_SERVICE_NAMESPACE}")

In [ ]:
from kubernetes import client, config

# Build InferenceService body for KServe (scikit-learn runtime)
body = {
    "apiVersion": "serving.kserve.io/v1beta1",
    "kind": "InferenceService",
    "metadata": {
        "name": INFERENCE_SERVICE_NAME,
        "namespace": INFERENCE_SERVICE_NAMESPACE,
        "annotations": {"serving.kserve.io/deploymentMode": "RawDeployment"},
    },
    "spec": {
        "predictor": {
            "model": {
                "modelFormat": {"name": "sklearn"},
                "storageUri": S3_MODEL_URI,
            }
        }
    },
}

print("InferenceService spec prepared.")

In [ ]:
# Load kubeconfig: in-cluster when running in pipeline pod, else local kubeconfig
try:
    config.load_incluster_config()
    print("Using in-cluster config.")
except config.ConfigException:
    try:
        config.load_kube_config()
        print("Using kubeconfig.")
    except config.ConfigException as e:
        raise RuntimeError("Cannot load Kubernetes config. Run inside cluster or set KUBECONFIG.") from e

In [ ]:
# Create or replace InferenceService via Kubernetes API
custom_api = client.CustomObjectsApi()
group, version = "serving.kserve.io", "v1beta1"
plural = "inferenceservices"

try:
    existing = custom_api.get_namespaced_custom_object(
        group, version, INFERENCE_SERVICE_NAMESPACE, plural, INFERENCE_SERVICE_NAME
    )
    # Update
    body["metadata"]["resourceVersion"] = existing["metadata"].get("resourceVersion", "")
    custom_api.replace_namespaced_custom_object(
        group, version, INFERENCE_SERVICE_NAMESPACE, plural, INFERENCE_SERVICE_NAME, body
    )
    print("InferenceService updated.")
except client.exceptions.ApiException as e:
    if e.status == 404:
        try:
            custom_api.create_namespaced_custom_object(
                group, version, INFERENCE_SERVICE_NAMESPACE, plural, body
            )
            print("InferenceService created.")
        except client.exceptions.ApiException as create_e:
            if create_e.status == 404:
                raise RuntimeError(
                    "KServe InferenceService CRD not found (404). "
                    "Ensure KServe/Serverless Serving is installed on the cluster (RHOAI has it by default)."
                ) from create_e
            raise
    else:
        raise

In [ ]:
# Verify deployment
obj = custom_api.get_namespaced_custom_object(
    group, version, INFERENCE_SERVICE_NAMESPACE, plural, INFERENCE_SERVICE_NAME
)
status = obj.get("status", {})
print(f"Status: {status.get('conditions', [{}])[-1].get('type', 'Unknown')}")
print(f"Ready: {status.get('conditions', [{}])[-1].get('status', 'Unknown')}")